In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib

# 1. 加载数据 (以 7天 为例，生成 28/56天时请修改此处)
sheet = '7all' 
df = pd.read_excel(r'updated_fc_predictions_with_fc_hat_all_data.xlsx', sheet_name=sheet)
df.dropna(inplace=True)

features = ['PC', 'PC_TYPE','FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
            'VOID', 'w/b', 'b/a','SCM%', 'CAGG%', 'FAGG%', 'FA%','SS%','SF%']

X = df[features].values
Y = df['fc (MPa)'].values

# 2. 划分数据集
Xtrain, Xtest, ytrain, ytest = train_test_split(X, Y, train_size=0.85, random_state=42)

# 3. 使用你提供的最佳参数训练 RF
best_params_rf = {
    'n_estimators': 500,
    'min_samples_split': 5,
    'min_samples_leaf': 1,
    'max_depth': None,
    'random_state': 42,
    'n_jobs': -1  # 使用所有核心加速
}

rf_model = RandomForestRegressor(**best_params_rf)
rf_model.fit(Xtrain, ytrain)

# 4. 准备 PDP 数据
mean_values = X.mean(axis=0)
wb_index = features.index('w/b')
wb_range = np.linspace(X[:, wb_index].min(), X[:, wb_index].max(), 100)

pdp_input = np.tile(mean_values, (wb_range.size, 1))
pdp_input[:, wb_index] = wb_range

# 5. 预测
pdp_predictions = rf_model.predict(pdp_input)

# 6. 保存结果 (记得对应日期修改文件名)
pdp_results = pd.DataFrame({
    'wb': wb_range, 
    'fc': pdp_predictions
})
output_filename = f'pdp_RF_wb_{sheet.replace("all", "")}.csv'
pdp_results.to_csv(output_filename, index=False)

print(f"✅ RF PDP 数据已保存至: {output_filename}")

✅ RF PDP 数据已保存至: pdp_RF_wb_7.csv
